# Exploration

Scratch notebook. Nothing here is load-bearing — anything that turns out to matter
should move into `src/`.

Questions worth answering here before the first real run:

1. How many genes actually survive the intersection across the chosen cohorts?
   If it collapses to a few hundred, the HVG selection has much less to work with
   than expected.
2. How different are library sizes between platforms? Visium and older Spatial
   Transcriptomics differ in spot size, so raw counts are not comparable — this is
   the batch effect we are trying to detect, so it is worth looking at directly.
3. Do the H&E patches look visually different between cohorts (stain, focus)?
   A human can usually spot the cohort immediately, which is a good prior for
   whether the model can too.

In [ ]:
import sys

sys.path.insert(0, "../src")

import matplotlib.pyplot as plt
import numpy as np

from data_loading import make_dummy_dataset, normalize_counts
from splits import build_all_splits

## Synthetic data

Same shape as the real thing, so the plotting code below can be written and
debugged before any download.

In [ ]:
data = make_dummy_dataset()
data.summary()

In [ ]:
# Library size by cohort, before normalisation. On real data this is the first
# place a platform effect shows up.
totals = data.counts.sum(axis=1)

fig, ax = plt.subplots(figsize=(6, 3))
for cohort, group in data.metadata.groupby("cohort_id"):
    ax.scatter(np.full(len(group), cohort), totals[group.index], alpha=0.6, s=12)
ax.set_ylabel("total counts per spot")
ax.set_title("Library size by cohort")
plt.tight_layout()

In [ ]:
# Sanity check the normalisation: every spot should sum to 10,000 before the log.
normalized = normalize_counts(data.counts)
np.expm1(normalized).sum(axis=1)[:5]

In [ ]:
splits = build_all_splits(data.metadata)
for strategy, folds in splits.items():
    for fold in folds:
        print(f"{fold.name:<32} {fold.sizes()}")

## Real HEST data

Needs `pip install -r requirements-data.txt` and a HEST download. Left unrun.

In [ ]:
# from data_loading import load_hest_metadata, select_starter_cohorts
#
# meta = load_hest_metadata("../data")
# select_starter_cohorts(meta)